In [1]:
import numpy as np

class NaivePCA_Adjusted:
    def __init__(self, n_components=None, pca_dict=None, explained_variance_=None):
        self.mean_ = None
        if n_components:
            self.n_components = n_components
            self.components_ = None
            self.explained_variance_ = None
        elif pca_dict and explained_variance_:
            self.pca_dict = pca_dict
            self.components_ = pca_dict
            self.explained_variance_ = explained_variance_
        else:
            raise ValueError("Either n_components or both pca_dict and explained_variance_ must be provided")
            
    def fit(self, X):
        # Center the data (so derived relationships are relevant among features)
        self.mean_ = np.mean(X, axis=0)
        X_centered = X - self.mean_
        
        # Calculate covariance matrix
        cov_matrix = np.cov(X_centered.T)
        
        # Perform eigenvalue decomposition
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
        
        # Sort eigenvalues and eigenvectors in descending order
        sorted_indices = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[sorted_indices]
        eigenvectors = eigenvectors[:, sorted_indices]
        
        # Store explained variance
        self.explained_variance_ = eigenvalues
        
        # Select top n_components
        if self.n_components is None:
            self.n_components = X.shape[1]
            
        self.components_ = eigenvectors[:, :self.n_components]
        
        return self

    def get_covariance_from_components(self):
        """
        Reconstructs the covariance matrix from components_ and explained_variance_
        
        Returns:
            Reconstructed covariance matrix
        """
        # Components are the eigenvectors
        V = self.components_.T  # Transpose to get eigenvectors as columns
        
        # Create diagonal matrix of eigenvalues
        Lambda = np.diag(self.explained_variance_)
    
        # Reconstruct covariance matrix: Σ = V Λ V^T
        # Where V is matrix of eigenvectors and Λ is diagonal matrix of eigenvalues
        cov_matrix = V @ Lambda @ V.T
    
        return cov_matrix
    
    def transform(self, X):
        if self.mean_ is None:
            raise ValueError("PCA has not been fitted yet")
            
        X_centered = X - self.mean_
        return np.dot(X_centered, self.components_)
        
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

### Part 1:

In [2]:
# Generate some sample data
np.random.seed(42)
n = 5   # Number of features (original dimension of data)
X = np.random.randn(100, n)  # 100 samples, n features

In [3]:
print(X[:5])

[[ 0.49671415 -0.1382643   0.64768854  1.52302986 -0.23415337]
 [-0.23413696  1.57921282  0.76743473 -0.46947439  0.54256004]
 [-0.46341769 -0.46572975  0.24196227 -1.91328024 -1.72491783]
 [-0.56228753 -1.01283112  0.31424733 -0.90802408 -1.4123037 ]
 [ 1.46564877 -0.2257763   0.0675282  -1.42474819 -0.54438272]]


In [4]:
# Function to generate PCA objects (designated on 'num_comp' number of components)
def generate_pca(num_comp):
    # Instantiate PCA objects which iteratively reduce the number of principal components by 1
    # Store the generated PCA objects in a dictionary ('pca')
    pca = {}
    for i in range(num_comp-1, 0, -1):      # Generate from num_comp-1 (less than original number of features) to 1
        current_pca = "pca_" + str(i)       # Base the specific PCA object name on the number of principal components
        pca[current_pca] = NaivePCA_Adjusted(n_components=i)  # Generate the PCA object
    return pca

pca_dict = generate_pca(n)

In [5]:
# Function to iteratively reduce the dimensions of original data (X) using the defined PCA objects
def transform_data(pca_dict, X):
    # Iterate over the PCA objects in reverse order (dims reduced in each iteration) to fit 
    # the previous PCA's (higher dimension) output and store the output in 
    # dictionary 'transformed_X'. Effectively, reduces the dimensions of the original data 
    # gradually to allow for fine-grained manipulation of the data on particular features
    transformed_X = {}
    
    # As mentioned, the following is just one of many possible implementations to iteratively reduced
    # the dimensions of the original data
    i = len(list(pca_dict.items()))     # set the starting index to the number of PCA objects
    while i >= 1:
        transformed_X[f"pca_{i}"] = pca_dict[f"pca_{i}"].fit_transform(X)
        X = transformed_X[f"pca_{i}"]
        i -= 1
    return transformed_X

fitted_data = transform_data(pca_dict, X)

In [6]:
for k, v in fitted_data.items():
    print(f"Shape of result through {k}: {v.shape}")

Shape of result through pca_4: (100, 4)
Shape of result through pca_3: (100, 3)
Shape of result through pca_2: (100, 2)
Shape of result through pca_1: (100, 1)


In [7]:
# View the components of each pca
for key, value in pca_dict.items():
    print(key)  # prints name relate to number of components (ex:pca_4, pca_3, pca_2, pca_1)
    print(value.components_)    # prints components of each pca

pca_4
[[ 0.36002699  0.00144754  0.42414635  0.23633513]
 [-0.45253955  0.56302745 -0.17517712 -0.49953211]
 [-0.25248232  0.59077075  0.55926507  0.42231723]
 [-0.18653311  0.1196727  -0.63230474  0.71218153]
 [-0.75302497 -0.56539045  0.27717531  0.09517877]]
pca_3
[[ 1.00000000e+00  2.81233612e-16 -3.89909722e-16]
 [-2.81233612e-16  1.00000000e+00  1.72084569e-15]
 [-3.89909722e-16  1.88737914e-15 -1.00000000e+00]
 [ 2.87039696e-16 -1.66533454e-15  2.88657986e-15]]
pca_2
[[ 1.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  1.00000000e+00]
 [ 0.00000000e+00 -4.64905892e-16]]
pca_1
[[1.]
 [0.]]


In [8]:
# Function to calculate the variance ratio (percent contribution of each feature for each principal component)
def variance_ratio(pca):
    explained_variance_ratio = {}
    for key, value in pca.items():
        explained_variance_ratio[key] = value.explained_variance_ / np.sum(value.explained_variance_)
    return explained_variance_ratio

explained_variance_ratio = variance_ratio(pca=pca_dict)
explained_variance_ratio

{'pca_4': array([0.26256655, 0.21579693, 0.20219416, 0.18127308, 0.13816928]),
 'pca_3': array([0.30466139, 0.25039364, 0.23461006, 0.21033491]),
 'pca_2': array([0.38581089, 0.3170884 , 0.29710071]),
 'pca_1': array([0.54888502, 0.45111498])}

In [9]:
for k, v in explained_variance_ratio.items():
    print(k, v)

pca_4 [0.26256655 0.21579693 0.20219416 0.18127308 0.13816928]
pca_3 [0.30466139 0.25039364 0.23461006 0.21033491]
pca_2 [0.38581089 0.3170884  0.29710071]
pca_1 [0.54888502 0.45111498]


### Part 2:

In [10]:
def threshold_explained_variance(threshold, ratio):
    threshold_metric = {}
    for k, v in ratio.items():
        elements = 0
        sum = 0.0
        for i in range(len(v)+1):
            if sum >= threshold:
                threshold_metric[k] = sum, elements
            else:
                sum += v[i]
                elements += 1       
    return threshold_metric

In [11]:
thresh_70 = threshold_explained_variance(threshold=0.70, ratio=explained_variance_ratio)
thresh_70

{'pca_4': (np.float64(0.8618307205417846), 4),
 'pca_3': (np.float64(0.789665092988757), 3),
 'pca_2': (np.float64(0.7028992868512434), 2),
 'pca_1': (np.float64(1.0), 2)}

In [12]:
thresh_80 = threshold_explained_variance(threshold=0.80, ratio=explained_variance_ratio)
thresh_80

{'pca_4': (np.float64(0.8618307205417846), 4),
 'pca_3': (np.float64(1.0), 4),
 'pca_2': (np.float64(1.0), 3),
 'pca_1': (np.float64(1.0), 2)}

In [13]:
thresh_90 = threshold_explained_variance(threshold=0.90, ratio=explained_variance_ratio)
thresh_90

{'pca_4': (np.float64(1.0), 5),
 'pca_3': (np.float64(1.0), 4),
 'pca_2': (np.float64(1.0), 3),
 'pca_1': (np.float64(1.0), 2)}

In [14]:
# ------------------------- EXPERIMENTAL -------------------------
def adjusted_var_pca_dict_explained_variance(pca_dict, thresh):
    num = [num_compo[1] for num_compo in thresh.values()]
    var_pca_explained_variance_ = {}
    var_pca_dict = {}
    for d, n in zip(pca_dict.items(), num):
        var_pca_explained_variance_.update({d[0] : d[1].explained_variance_[:n]})
        var_pca_dict.update({d[0] : d[1].components_[:n]})
    return var_pca_explained_variance_, var_pca_dict

var_pca_explained_variance_, var_pca_dict = adjusted_var_pca_dict_explained_variance(pca_dict, thresh_70)

In [15]:
print(var_pca_explained_variance_)
print(var_pca_dict)

{'pca_4': array([1.26528496, 1.03990633, 0.97435577, 0.87353896]), 'pca_3': array([1.26528496, 1.03990633, 0.97435577]), 'pca_2': array([1.26528496, 1.03990633]), 'pca_1': array([1.26528496, 1.03990633])}
{'pca_4': array([[ 0.36002699,  0.00144754,  0.42414635,  0.23633513],
       [-0.45253955,  0.56302745, -0.17517712, -0.49953211],
       [-0.25248232,  0.59077075,  0.55926507,  0.42231723],
       [-0.18653311,  0.1196727 , -0.63230474,  0.71218153]]), 'pca_3': array([[ 1.00000000e+00,  2.81233612e-16, -3.89909722e-16],
       [-2.81233612e-16,  1.00000000e+00,  1.72084569e-15],
       [-3.89909722e-16,  1.88737914e-15, -1.00000000e+00]]), 'pca_2': array([[1., 0.],
       [0., 1.]]), 'pca_1': array([[1.],
       [0.]])}


In [16]:
sliced_dict = NaivePCA_Adjusted(pca_dict=var_pca_dict, explained_variance_=var_pca_explained_variance_)
sliced_dict

In [17]:
for k, v in pca_dict.items():
    print(k, v.explained_variance_)
    print(k, v.components_)
    print("\n")

pca_4 [1.26528496 1.03990633 0.97435577 0.87353896 0.66582554]
pca_4 [[ 0.36002699  0.00144754  0.42414635  0.23633513]
 [-0.45253955  0.56302745 -0.17517712 -0.49953211]
 [-0.25248232  0.59077075  0.55926507  0.42231723]
 [-0.18653311  0.1196727  -0.63230474  0.71218153]
 [-0.75302497 -0.56539045  0.27717531  0.09517877]]


pca_3 [1.26528496 1.03990633 0.97435577 0.87353896]
pca_3 [[ 1.00000000e+00  2.81233612e-16 -3.89909722e-16]
 [-2.81233612e-16  1.00000000e+00  1.72084569e-15]
 [-3.89909722e-16  1.88737914e-15 -1.00000000e+00]
 [ 2.87039696e-16 -1.66533454e-15  2.88657986e-15]]


pca_2 [1.26528496 1.03990633 0.97435577]
pca_2 [[ 1.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  1.00000000e+00]
 [ 0.00000000e+00 -4.64905892e-16]]


pca_1 [1.26528496 1.03990633]
pca_1 [[1.]
 [0.]]




In [18]:
sliced_dict.explained_variance_

{'pca_4': array([1.26528496, 1.03990633, 0.97435577, 0.87353896]),
 'pca_3': array([1.26528496, 1.03990633, 0.97435577]),
 'pca_2': array([1.26528496, 1.03990633]),
 'pca_1': array([1.26528496, 1.03990633])}

In [19]:
sliced_dict.components_

{'pca_4': array([[ 0.36002699,  0.00144754,  0.42414635,  0.23633513],
        [-0.45253955,  0.56302745, -0.17517712, -0.49953211],
        [-0.25248232,  0.59077075,  0.55926507,  0.42231723],
        [-0.18653311,  0.1196727 , -0.63230474,  0.71218153]]),
 'pca_3': array([[ 1.00000000e+00,  2.81233612e-16, -3.89909722e-16],
        [-2.81233612e-16,  1.00000000e+00,  1.72084569e-15],
        [-3.89909722e-16,  1.88737914e-15, -1.00000000e+00]]),
 'pca_2': array([[1., 0.],
        [0., 1.]]),
 'pca_1': array([[1.],
        [0.]])}

In [20]:
print(f"Shape of Fitted Data through PCA_4: {len(fitted_data['pca_4'])}x{len(fitted_data['pca_4'][0])}")
print(f"Shape of Fitted Data through PCA_3: {len(fitted_data['pca_3'])}x{len(fitted_data['pca_3'][0])}")
print(f"Shape of Fitted Data through PCA_2: {len(fitted_data['pca_2'])}x{len(fitted_data['pca_2'][0])}")
print(f"Shape of Fitted Data through PCA_1: {len(fitted_data['pca_1'])}x{len(fitted_data['pca_1'][0])}")

Shape of Fitted Data through PCA_4: 100x4
Shape of Fitted Data through PCA_3: 100x3
Shape of Fitted Data through PCA_2: 100x2
Shape of Fitted Data through PCA_1: 100x1


In [21]:
def data_fit_on_var_pca(fitted_data, sliced_dict):
    fitted_data_on_var = {}
    for i in reversed(range(1,5)):
        fitted_data_on_var[f"pca_{i}"] = np.dot(fitted_data[f"pca_{i}"], sliced_dict.components_[f"pca_{i}"].T)
    return fitted_data_on_var

data_through_var = data_fit_on_var_pca(fitted_data, sliced_dict)
for k, v in data_through_var.items():
    print(k, v[:5])

pca_4 [[ 0.25113796 -0.37737861  0.85454569  1.31863746]
 [-0.36507263  1.40413095  0.92967783 -0.64372466]
 [ 0.68617249  0.07442589 -0.09413023 -1.75084539]
 [ 0.65286922 -0.43605329 -0.04736137 -0.72834999]
 [ 1.0256758  -0.57347089  0.35003774 -1.68025279]]
pca_3 [[-0.00302966  0.59882437 -0.25593298]
 [-1.28683249  0.95790102  0.67534202]
 [ 1.66549521  0.60500276  0.92687858]
 [ 1.4363139   0.28308313  0.47223014]
 [ 1.31536447  0.03084568  1.62380989]]
pca_2 [[-0.00302966  0.59882437]
 [-1.28683249  0.95790102]
 [ 1.66549521  0.60500276]
 [ 1.4363139   0.28308313]
 [ 1.31536447  0.03084568]]
pca_1 [[-0.00302966  0.        ]
 [-1.28683249  0.        ]
 [ 1.66549521  0.        ]
 [ 1.4363139   0.        ]
 [ 1.31536447  0.        ]]


### Part 3:

In [22]:
import copy # Utilize to create complete independent copy of pca_dict

weights_dict = {'pca_3' : {2: 7}, 'pca_4' : {0: 2}}

def adjusted_weighted_components(pca_dict, weights_dict):
    adjusted_dict = copy.deepcopy(pca_dict) # Since its value is itself a dictionary and mutable (use as to not affect original pca_dict)
    try:
        for k, v in pca_dict.items():
            for key, value in weights_dict.items():
                if k == key:
                    adjusted_dict[k].components_[list(value.keys())] = v.components_[list(value.keys())][0] * list(value.values())[0]
    except IndexError as e:
        print(f"***ERROR: {e}. Check that the indices are valid for the specific number of components.***")
    return adjusted_dict

In [23]:
adjusted_weighted_dict = adjusted_weighted_components(pca_dict=pca_dict, weights_dict=weights_dict)
for k, v in adjusted_weighted_dict.items():
    print(k, v.components_)

pca_4 [[ 0.72005397  0.00289508  0.84829269  0.47267025]
 [-0.45253955  0.56302745 -0.17517712 -0.49953211]
 [-0.25248232  0.59077075  0.55926507  0.42231723]
 [-0.18653311  0.1196727  -0.63230474  0.71218153]
 [-0.75302497 -0.56539045  0.27717531  0.09517877]]
pca_3 [[ 1.00000000e+00  2.81233612e-16 -3.89909722e-16]
 [-2.81233612e-16  1.00000000e+00  1.72084569e-15]
 [-2.72936805e-15  1.32116540e-14 -7.00000000e+00]
 [ 2.87039696e-16 -1.66533454e-15  2.88657986e-15]]
pca_2 [[ 1.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  1.00000000e+00]
 [ 0.00000000e+00 -4.64905892e-16]]
pca_1 [[1.]
 [0.]]


The explained_variance_ should be the same for which relates the original components to the original data. Since the explained_variance_ is a measure of the variance of the data, for which the components capture, this inherent signifier should remain the same. Applying weights to the components is an entirely different process onto itself. It's essentially done for niche analysis and 'expert' manipulation. The process serves a fundamentally different purpose than the basic, naive PCA.

In [24]:
def reconstruct_cov_matrix(pca_dict):
    transformed_X = {}

    i = len(list(pca_dict.items()))     
    while i >= 1:
        transformed_X[f"pca_{i}"] = pca_dict[f"pca_{i}"].get_covariance_from_components()
        i -= 1
    return transformed_X

reconstructed_cov_matrix = reconstruct_cov_matrix(pca_dict=adjusted_weighted_dict)
reconstructed_cov_matrix

{'pca_4': array([[ 1.33904773, -0.14367911,  0.68177079,  0.39805655],
        [-0.14367911,  0.89507326,  0.05202318, -0.0090275 ],
        [ 0.68177079,  0.05202318,  1.64756959,  0.45265812],
        [ 0.39805655, -0.0090275 ,  0.45265812,  1.16504767]]),
 'pca_3': array([[ 1.26528496e+00,  6.33840464e-17,  1.81222817e-14],
        [ 6.33840464e-17,  1.03990633e+00, -8.83204408e-14],
        [ 1.81222817e-14, -8.83204408e-14,  4.77434328e+01]]),
 'pca_2': array([[1.26528496, 0.        ],
        [0.        , 1.03990633]]),
 'pca_1': array([[1.26528496]])}

In [25]:
def data_fit_on_weights_pca(fitted_data, adjusted_dict):
    fitted_data_on_var = {}
    for i in reversed(range(1,5)):
        fitted_data_on_var[f"pca_{i}"] = np.dot(fitted_data[f"pca_{i}"], adjusted_dict[f"pca_{i}"].components_.T)
    return fitted_data_on_var
data_through_weights = data_fit_on_weights_pca(fitted_data, adjusted_dict=adjusted_weighted_dict)

In [26]:
for k, v in data_through_weights.items():
    print(k, v[:5])

pca_4 [[ 0.50227591 -0.37737861  0.85454569  1.31863746 -0.26227866]
 [-0.73014526  1.40413095  0.92967783 -0.64372466  0.53825632]
 [ 1.37234497  0.07442589 -0.09413023 -1.75084539 -1.46313661]
 [ 1.30573843 -0.43605329 -0.04736137 -0.72834999 -1.13689817]
 [ 2.0513516  -0.57347089  0.35003774 -1.68025279 -0.61290239]]
pca_3 [[-3.02966342e-03  5.98824374e-01 -1.79153083e+00 -2.59341570e-16]
 [-1.28683249e+00  9.57901022e-01  4.72739414e+00 -3.91402634e-15]
 [ 1.66549521e+00  6.05002761e-01  6.48815009e+00 -3.20497781e-15]
 [ 1.43631390e+00  2.83083135e-01  3.30561095e+00 -1.42227902e-15]
 [ 1.31536447e+00  3.08456759e-02  1.13666692e+01 -4.36106348e-15]]
pca_2 [[-3.02966342e-03  5.98824374e-01 -2.78396979e-16]
 [-1.28683249e+00  9.57901022e-01 -4.45333829e-16]
 [ 1.66549521e+00  6.05002761e-01 -2.81269348e-16]
 [ 1.43631390e+00  2.83083135e-01 -1.31607017e-16]
 [ 1.31536447e+00  3.08456759e-02 -1.43403365e-17]]
pca_1 [[-0.00302966  0.        ]
 [-1.28683249  0.        ]
 [ 1.66549521 

As expected the data_through_weights variable contains 5 features (original number), down to 2 features (as indicated by the original definitions of the 4 PCA constructs). This is different from data_through_var, which is constructed by limiting the retained features in each PCA construct.

In [27]:
from collections import defaultdict

def aggregate_pca(pca_1, pca_2):
    agg_pca = defaultdict(list)
    for (k_s, v_s), (k_a, v_a) in zip(pca_1.components_.items(), pca_2.items()):
        if len(k_s) == len(k_a):
            min_compo = min(len(v_s), len(v_a.components_))
            agg_pca[k_s].append((v_s[:min_compo] + v_a.components_[:min_compo]) / 2)
    return agg_pca

aggre_pca = aggregate_pca(pca_1=sliced_dict, pca_2=adjusted_weighted_dict)

In [28]:
def aggregate_explained_variance(pca_1, pca_2):
    agg_explained_variance = defaultdict(list)
    for (k_s, v_s), (k_a, v_a) in zip(pca_1.explained_variance_.items(), pca_2.items()):
        if len(k_s) == len(k_a):
            min_compo = min(len(v_s), len(v_a.explained_variance_))
            agg_explained_variance[k_s].append((v_s[:min_compo] + v_a.explained_variance_[:min_compo]) / 2)
    return agg_explained_variance

aggre_explained_variance_ = aggregate_explained_variance(pca_1=sliced_dict, pca_2=adjusted_weighted_dict)

Aggregated Explained Variance function is not really needed in this case (from consequence of how the aggregation occurs and criteria on feature dependency). However, it was included for possible future cases in which manipulation may be required.

In [29]:
for k, v in aggre_explained_variance_.items():
    print(k, v[0])

pca_4 [1.26528496 1.03990633 0.97435577 0.87353896]
pca_3 [1.26528496 1.03990633 0.97435577]
pca_2 [1.26528496 1.03990633]
pca_1 [1.26528496 1.03990633]


In [30]:
for k, v in aggre_pca.items():
    print(k, v[0])

pca_4 [[ 0.54004048  0.00217131  0.63621952  0.35450269]
 [-0.45253955  0.56302745 -0.17517712 -0.49953211]
 [-0.25248232  0.59077075  0.55926507  0.42231723]
 [-0.18653311  0.1196727  -0.63230474  0.71218153]]
pca_3 [[ 1.00000000e+00  2.81233612e-16 -3.89909722e-16]
 [-2.81233612e-16  1.00000000e+00  1.72084569e-15]
 [-1.55963889e-15  7.54951657e-15 -4.00000000e+00]]
pca_2 [[1. 0.]
 [0. 1.]]
pca_1 [[1.]
 [0.]]


Proposed workflow rather than strict methodology. Due to the intricacies of data, with further dependencies on their constituents, most of these subroutines need to be highly tailored. As such, there doesn't exist a "main accepted workflow". The demonstrations presented highlight various methods in which analysis may be conducted as a means to extract relevant insights. 